In [1]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'F:\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline
from transformers.utils import TRANSFORMERS_CACHE

print(f"实际缓存路径: {TRANSFORMERS_CACHE}")

# 验证路径是否正确
if cache_dir == TRANSFORMERS_CACHE:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {TRANSFORMERS_CACHE}")

# 运行情感分析
classifier = pipeline("sentiment-analysis")
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

print(f"\n分析结果: {result}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

import os

# 检查并修复缓存目录权限
cache_dir = r"F:\Anaconda\envs\pytorch\cache"
if os.path.exists(cache_dir):
    import subprocess
    # 给当前用户完全控制权限
    subprocess.run(['icacls', cache_dir, '/grant', 'Everyone:F', '/T'], shell=True)

设置的缓存目录: F:\Anaconda\envs\pytorch\cache


f:\Anaconda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
f:\Anaconda\envs\pytorch\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


实际缓存路径: F:\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！



Device set to use cuda:0



分析结果: [{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--glue
  📁 evaluate
  📁 metrics
  📁 models--bert-base-uncased
  📁 models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
  📁 modules
  📁 xet


预处理

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

删除与模型不需要的列（如 sentence1 和 sentence2 列）。

将列名 label 重命名为 labels （因为模型默认的输入是 labels ）。

设置数据集的格式，使其返回 PyTorch 张量而不是列表。

In [3]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

define our dataloaders

In [4]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

为了快速检验数据处理中没有错误，我们可以这样检验其中的一个 batch：

In [5]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 71]),
 'token_type_ids': torch.Size([8, 71]),
 'attention_mask': torch.Size([8, 71])}

Note that the actual shapes will probably be slightly different for you since we set shuffle=True for the training dataloader and we are padding to the maximum length inside the batch.

Now that we’re completely finished with data preprocessing (a satisfying yet elusive goal for any ML practitioner), let’s turn to the model. We instantiate it exactly as we did in the previous section:

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


To make sure that everything will go smoothly during training, we pass our batch to this model

In [7]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6876, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


an optimizer and a learning rate scheduler are needed too

Trainer 使用的优化器是 AdamW ，它与 Adam 相同，但加入了权重衰减正则化的一点变化

In [8]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

AdamW with weight decay: AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

8-bit Adam: Use bitsandbytes for memory-efficient optimization

Different learning rates: Lower learning rates (1e-5 to 3e-5) often work better for large models

In [9]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377


In [10]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

device(type='cuda')

We are now ready to train! To get some sense of when training will be finished, we add a progress bar over our number of training steps, using the tqdm library:

In [11]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()#更新参数
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)#更新进度条1布

100%|██████████| 1377/1377 [05:10<00:00,  4.05it/s]

Gradient Clipping: Add torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) before optimizer.step()

Mixed Precision: Use torch.cuda.amp.autocast() and GradScaler for faster training

Gradient Accumulation: Accumulate gradients over multiple batches to simulate larger batch sizes

Checkpointing: Save model checkpoints periodically to resume training if interrupted

梯度裁剪	稳定训练，防止梯度爆炸	训练RNN、Transformer等易梯度爆炸的模型

混合精度	加速训练，减少显存	需要快速训练的大模型，显存有限时

梯度累积	模拟大批次训练	显存不足，但需要大批次效果时

检查点	容错恢复，模型选择	长时间训练，需要中断恢复或选择最佳模型

The evaluation loop

In [13]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)#获取最后一个维度（类别维度）的最大值索引
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8602941176470589, 'f1': 0.9035532994923858}

Supercharge your training loop with Accelerate

using the Accelerate library, with just a few adjustments we can enable distributed training on multiple GPUs or TPUs.

In [15]:
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 1377/1377 [05:17<00:00,  4.19it/s]

Accelerate 的完整训练循环,单独放再.py文件即可调用

In [ ]:
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

终端中调用accelerate config

选择完成后
最后输入
accelerate launch train.py

colab中只需要整个放入training_function

from accelerate import notebook_launcher

notebook_launcher(training_function)

Now that you’ve learned how to implement training from scratch, here are some additional considerations for production use:

Model Evaluation: Always evaluate your model on multiple metrics, not just accuracy. Use the 🤗 Evaluate library for comprehensive evaluation.

Hyperparameter Tuning: Consider using libraries like Optuna or Ray Tune for systematic hyperparameter optimization.

Model Monitoring: Track training metrics, learning curves, and validation performance throughout training.

Model Sharing: Once trained, share your model on the Hugging Face Hub to make it available to the community.

Efficiency: For large models, consider techniques like gradient checkpointing, parameter-efficient fine-tuning (LoRA, AdaLoRA), or quantization methods.

Manual training loops give you complete control but require understanding of the proper sequence: forward → backward → optimizer step → scheduler step → zero gradients

AdamW with weight decay is the recommended optimizer for transformer models

Always use model.eval() and torch.no_grad() during evaluation for correct behavior and efficiency

Accelerate makes distributed training accessible with minimal code changes

Device management (moving tensors to GPU/CPU) is crucial for PyTorch operations

Modern techniques like mixed precision, gradient accumulation, and gradient clipping can significantly improve training efficiency